# Try implementation of PhosX

In [41]:
import pandas as pd

from utils import *

In [42]:
df = pd.read_csv("Experiment/hme1_mutants_test/Data/Processed/Mutant_cell_lines_test.tsv", sep= "\t")

# Remove rows without PTM
df_filtered = df[df.Assigned_Modifications_clean.notna()]
print(f"Original size: {df.shape}")

# Drop rows where site is not detected in the starvation control
df_filtered = df_filtered[df_filtered["WT_raw:abs_EGF_starve_r1"] != 0]
df_filtered = df_filtered[df_filtered["BRAFS151A_raw:abs_EGF_starve_r1"] != 0]
df_filtered = df_filtered[df_filtered["GAB1Y259A_raw:abs_EGF_starve_r1"] != 0]

raw_columns = [element for element in df.columns if 'raw' in element]

print(f"Dataframe after removing sites with no starve detected: {df_filtered.shape}")
replicates = [element for element in raw_columns if "starve" not in element]
# print(f"Raw columns of time points replicates: {replicates}")

replicates = list(set([element.replace("_r1", "").replace("_r2", "") for element in replicates]))

suffixes = ["_r1", "_r2"]

# Drop rows if the phosphosite was not detected in the 2 replicates for any time point and any condition
for condition in replicates:
    mask = (
        (df_filtered[f"{condition}{suffixes[0]}"] == 0) &
        (df_filtered[f"{condition}{suffixes[1]}"] == 0)
    )
    df_filtered = df_filtered.drop(df_filtered[mask].index)

print(f"Dataframe after removing sites where a site is missing in the two replivates: {df_filtered.shape}")

df_filtered = filter_dynamics_extremes_mutants(df = df_filtered,
                                       data_type="log2:FC",
                                       threshold=0.5,
                                       exclude_full = False,
                                       conditions = ["_EGF_"],
                                       cell_lines = ["WT", "BRAFS151A", "GAB1Y259A"],)
print(f"Dataframe after selecting log2:FC > 0.5: {df_filtered.shape}")

Original size: (175089, 136)
Dataframe after removing sites with no starve detected: (13474, 136)
Dataframe after removing sites where a site is missing in the two replivates: (10885, 136)
Dataframe after selecting log2:FC > 0.5: (9561, 136)


In [53]:
test_df = df_filtered.head(2000).copy()
test_df

,protein_Id,protein_name,description,Peptide_Sequence,Modified_Sequence,site_start,site_end,Peptide_Length,Charges,Assigned_Modifications,...,WT_log2:FC_EGF_10,WT_log2:FC_EGF_25,BRAFS151A_log2:FC_EGF_starve,BRAFS151A_log2:FC_EGF_2,BRAFS151A_log2:FC_EGF_10,BRAFS151A_log2:FC_EGF_25,GAB1Y259A_log2:FC_EGF_starve,GAB1Y259A_log2:FC_EGF_2,GAB1Y259A_log2:FC_EGF_10,GAB1Y259A_log2:FC_EGF_25
17,Q9NQS7,INCENP,Inner centromere protein,AAAAAAAATMALAAPSSPTPESPTMLTK,AAAAAAAATMALAAPS[79.9663]SPTPES[79.9663]PTMLTK,127,154,28,"2,3","16S(79.9663),22S(79.9663)",...,-0.808449,-0.179979,0.0,0.888322,0.803156,0.130157,0.0,-0.955918,0.016177,0.179395
38,Q6SPF0,SAMD1,Sterile alpha motif domain-containing protein 1,AAAAAATAPPSPGPAQPGPR,AAAAAATAPPS[79.9663]PGPAQPGPR,151,170,20,2,11S(79.9663),...,0.293958,0.354369,0.0,-0.962931,-0.290644,-0.431276,0.0,-0.080581,-0.012117,-0.075329
43,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAMET...,177,217,41,3,30S(79.9663),...,0.023431,0.330201,0.0,-2.574105,-0.605887,-1.345394,0.0,-0.221699,-0.409022,0.028441
44,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAM[1...,177,217,41,3,"30S(79.9663),35M(15.9949)",...,0.157587,0.527729,0.0,-1.980528,-0.572586,-1.853071,0.0,0.013684,-0.283210,-0.064792
45,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPAS[79.9663]EDEDDEDDEDDEDDDDDEEDDSEEEAMET...,177,217,41,3,8S(79.9663),...,0.521452,0.545618,0.0,-2.079285,-0.628950,-2.164579,0.0,-0.291065,-0.513799,0.083652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36223,P55060,CSE1L,Exportin-2,FFEGPVTGIFSGYVNSMLQEYAK,FFEGPVTGIFSGYVNSM[15.9949]LQEYAK,396,418,23,2,17M(15.9949),...,0.455623,-0.902055,0.0,-0.884990,-1.112821,-0.756359,0.0,-1.200886,-0.301692,-1.021926
36227,P15056,BRAF,Serine/threonine-protein kinase B-raf,FFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPSK,FFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPS[79....,294,338,45,"3,4",42S(79.9663),...,-0.040612,0.298353,0.0,0.781576,0.719423,0.781943,0.0,-0.112800,0.015319,-0.139216
36323,Q7Z2W4,ZC3HAV1,Zinc finger CCCH-type antiviral protein 1,FFQGSQEFLASASASAER,FFQGS[79.9663]QEFLASASASAER,253,270,18,2,5S(79.9663),...,-0.779464,0.017637,0.0,0.740177,0.942132,0.037483,0.0,0.524610,-0.307874,-0.067295
36442,Q96Q89,KIF20B,Kinesin-like protein KIF20B,FGDFLQHSPSILQSK,FGDFLQHS[79.9663]PSILQSK,1733,1747,15,"2,3",8S(79.9663),...,-0.478216,-0.209172,0.0,1.516128,0.632705,0.854183,0.0,-0.020795,-0.204732,-0.288416


## Data pre-processing

In [54]:
test_df["phosx"] = (test_df["Modified_Sequence"].str.replace("S[79.9663]", "s", regex=False)
                    .str.replace("T[79.9663]", "t", regex=False)
                    .str.replace("Y[79.9663]", "y", regex=False)
                    .str.replace("[15.9949]", "", regex=False))

test_df = test_df.loc[test_df["n_localized"] == 1]
test_df = test_df.loc[test_df["other_localized"] == 0]
test_df

,protein_Id,protein_name,description,Peptide_Sequence,Modified_Sequence,site_start,site_end,Peptide_Length,Charges,Assigned_Modifications,...,WT_log2:FC_EGF_25,BRAFS151A_log2:FC_EGF_starve,BRAFS151A_log2:FC_EGF_2,BRAFS151A_log2:FC_EGF_10,BRAFS151A_log2:FC_EGF_25,GAB1Y259A_log2:FC_EGF_starve,GAB1Y259A_log2:FC_EGF_2,GAB1Y259A_log2:FC_EGF_10,GAB1Y259A_log2:FC_EGF_25,phosx
38,Q6SPF0,SAMD1,Sterile alpha motif domain-containing protein 1,AAAAAATAPPSPGPAQPGPR,AAAAAATAPPS[79.9663]PGPAQPGPR,151,170,20,2,11S(79.9663),...,0.354369,0.0,-0.962931,-0.290644,-0.431276,0.0,-0.080581,-0.012117,-0.075329,AAAAAATAPPsPGPAQPGPR
43,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAMET...,177,217,41,3,30S(79.9663),...,0.330201,0.0,-2.574105,-0.605887,-1.345394,0.0,-0.221699,-0.409022,0.028441,AAAAAPASEDEDDEDDEDDEDDDDDEEDDsEEEAMETTPAK
45,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPAS[79.9663]EDEDDEDDEDDEDDDDDEEDDSEEEAMET...,177,217,41,3,8S(79.9663),...,0.545618,0.0,-2.079285,-0.628950,-2.164579,0.0,-0.291065,-0.513799,0.083652,AAAAAPAsEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK
92,Q9UPT8,ZC3H4,Zinc finger CCCH domain-containing protein 4,AAAAPAATTATPPPEGAPPQPGVHNLPVPTLFGTVK,AAAAPAATTAT[79.9663]PPPEGAPPQPGVHNLPVPTLFGTVK,1225,1260,36,"3,4",11T(79.9663),...,0.104813,0.0,-0.717353,-0.237849,-0.299500,0.0,-0.223029,-0.256941,-0.358785,AAAAPAATTAtPPPEGAPPQPGVHNLPVPTLFGTVK
103,P52701,MSH6,DNA mismatch repair protein Msh6,AAAAPGASPSPGGDAAWSEAGPGPR,AAAAPGAS[79.9663]PSPGGDAAWSEAGPGPR,34,58,25,"2,3",8S(79.9663),...,-0.211024,0.0,0.713253,0.208418,0.370986,0.0,-0.031502,-0.827080,-0.525412,AAAAPGAsPSPGGDAAWSEAGPGPR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36208,Q6P4E1,GOLM2,Protein GOLM2,FFDENESPVDPQHGSK,FFDENES[79.9663]PVDPQHGSK,360,375,16,"2,3",7S(79.9663),...,0.310100,0.0,0.776099,1.091460,1.027165,0.0,0.664123,0.248222,0.188103,FFDENEsPVDPQHGSK
36227,P15056,BRAF,Serine/threonine-protein kinase B-raf,FFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPSPSK,FFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPS[79....,294,338,45,"3,4",42S(79.9663),...,0.298353,0.0,0.781576,0.719423,0.781943,0.0,-0.112800,0.015319,-0.139216,FFEHHPIPQEEASLAETALTSGSSPSAPASDSIGPQILTSPsPSK
36323,Q7Z2W4,ZC3HAV1,Zinc finger CCCH-type antiviral protein 1,FFQGSQEFLASASASAER,FFQGS[79.9663]QEFLASASASAER,253,270,18,2,5S(79.9663),...,0.017637,0.0,0.740177,0.942132,0.037483,0.0,0.524610,-0.307874,-0.067295,FFQGsQEFLASASASAER
36442,Q96Q89,KIF20B,Kinesin-like protein KIF20B,FGDFLQHSPSILQSK,FGDFLQHS[79.9663]PSILQSK,1733,1747,15,"2,3",8S(79.9663),...,-0.209172,0.0,1.516128,0.632705,0.854183,0.0,-0.020795,-0.204732,-0.288416,FGDFLQHsPSILQSK


In [55]:
def extract_window(seq, left=5, right=4):
    for i, char in enumerate(seq):
        if char.islower():
            start = max(0, i - left)
            end = min(len(seq), i + right + 1)
            window_seq = seq[start:end]
            left_pad = "_" * max(0, left - i)
            right_pad = "_" * max(0, (i + right + 1) - len(seq))
            return (left_pad + window_seq + right_pad).upper()
    return None

test_df["window"] = test_df["phosx"].apply(extract_window)

test_df[["phosx", "window"]].head(10)

,phosx,window
38,AAAAAATAPPsPGPAQPGPR,ATAPPSPGPA
43,AAAAAPASEDEDDEDDEDDEDDDDDEEDDsEEEAMETTPAK,DEEDDSEEEA
45,AAAAAPAsEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAPASEDED
92,AAAAPAATTAtPPPEGAPPQPGVHNLPVPTLFGTVK,AATTATPPPE
103,AAAAPGAsPSPGGDAAWSEAGPGPR,AAPGASPSPG
154,AAALQALQAQAPtSPPPPPPPLK,QAQAPTSPPP
191,AAATGNAsPGKLEHSK,ATGNASPGKL
198,AAAtPESQEPQAK,__AAATPESQ
212,AAAYDIsEDEED,AAYDISEDEE
224,AADEDWDsELEDDLLGEDLLSGKK,DEDWDSELED


In [60]:
phosx_df = test_df[["window", "WT_log2:FC_EGF_10"]].copy()
phosx_df["WT_log2:FC_EGF_10"] = phosx_df["WT_log2:FC_EGF_10"].astype(float)
phosx_df
# phosx_df = phosx_df.to_csv("PhosX/testing.seqrnk", sep= "\t", index = False, header = False)

,window,WT_log2:FC_EGF_10
38,ATAPPSPGPA,0.293958
43,DEEDDSEEEA,0.023431
45,AAAPASEDED,0.521452
92,AATTATPPPE,-0.382222
103,AAPGASPSPG,0.025595
...,...,...
36208,FDENESPVDP,-0.048464
36227,ILTSPSPSK_,-0.040612
36323,_FFQGSQEFL,-0.779464
36442,DFLQHSPSIL,-0.478216


 phosx testing.seqrnk  > kinase_activity.tmp

In [61]:
# import phosx
#
# results = phosx(phosx_df) # phosx("PhosX/testing.seqrnk")
#
# # results is a dict with:
# results["kinase_activity"]       # differential activity scores per kinase
# results["st_assigned_substrates"] # Ser/Thr kinase → substrate assignments
# results["y_assigned_substrates"]  # Tyr kinase → substrate assignments


TypeError: 'module' object is not callable